# 🦕 DINO SDK v1.2.0 - Teste no Databricks

Este notebook demonstra como configurar e usar o DINO SDK v1.2.0 corretamente no ambiente Databricks.

## 📋 Objetivos:
1. Detectar o ambiente Databricks
2. Configurar a sessão Spark adequadamente
3. Inicializar o DINO SDK com configurações corretas
4. Validar que tudo está funcionando
5. Criar schemas no Unity Catalog

## 1. Instalação do DINO SDK

Primeiro, vamos instalar o DINO SDK v1.2.0 e reiniciar o ambiente Python.

In [ ]:
# Instalar DINO SDK v1.2.0
%pip install /dbfs/FileStore/shared_uploads/dino_sdk-1.2.0-py3-none-any.whl --quiet

# Reiniciar ambiente Python para carregar as mudanças
dbutils.library.restartPython()

## 2. Detectar Ambiente Databricks

Vamos verificar se estamos em um ambiente Databricks e se as variáveis necessárias estão disponíveis.

In [ ]:
# Detectar ambiente Databricks
print("🔍 Detectando Ambiente Databricks")
print("=" * 40)

# Verificar se estamos no Databricks
databricks_detected = False

# Método 1: Verificar se dbutils existe
try:
    dbutils.fs.ls('/')
    print("✅ dbutils detectado - ambiente Databricks confirmado")
    databricks_detected = True
except:
    print("❌ dbutils não encontrado")

# Método 2: Verificar se spark está disponível globalmente
try:
    spark_version = spark.version
    print(f"✅ Spark detectado - versão: {spark_version}")
    print(f"✅ Spark App Name: {spark.sparkContext.appName}")
    print(f"✅ Spark Master: {spark.sparkContext.master}")
except:
    print("❌ Variável 'spark' não encontrada globalmente")

# Método 3: Verificar contexto Databricks específico
try:
    workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
    print(f"✅ Workspace URL: {workspace_url}")
except:
    print("❌ Contexto Databricks não encontrado")

if databricks_detected:
    print("\n🎉 Ambiente Databricks confirmado!")
else:
    print("\n⚠️ Ambiente Databricks não detectado")

## 3. Configurar Contexto Spark para Databricks

Vamos configurar o contexto Spark especificamente para o ambiente Databricks.

In [ ]:
# Configurar contexto Spark para Databricks
print("⚙️ Configurando Contexto Spark")
print("=" * 35)

# Verificar se a variável spark está disponível
def get_spark_session():
    """Função para obter a sessão Spark no Databricks"""
    
    # Método 1: Usar variável global spark (padrão Databricks)
    try:
        if 'spark' in globals():
            session = globals()['spark']
            print("✅ Sessão Spark obtida via variável global")
            return session
    except:
        pass
    
    # Método 2: Usar PySpark
    try:
        from pyspark.sql import SparkSession
        session = SparkSession.getActiveSession()
        if session:
            print("✅ Sessão Spark obtida via SparkSession.getActiveSession()")
            return session
    except:
        pass
    
    # Método 3: Criar nova sessão (último recurso)
    try:
        from pyspark.sql import SparkSession
        session = SparkSession.builder.getOrCreate()
        print("✅ Nova sessão Spark criada")
        return session
    except Exception as e:
        print(f"❌ Erro ao criar sessão Spark: {e}")
        return None

# Obter sessão Spark
spark_session = get_spark_session()

if spark_session:
    print(f"✅ Sessão Spark configurada com sucesso!")
    print(f"   App Name: {spark_session.sparkContext.appName}")
    print(f"   Spark Version: {spark_session.version}")
    
    # Testar uma query simples
    try:
        result = spark_session.sql("SELECT 1 as test").collect()
        print(f"✅ Teste de query bem-sucedido: {result[0]['test']}")
    except Exception as e:
        print(f"❌ Erro no teste de query: {e}")
else:
    print("❌ Não foi possível configurar sessão Spark")

## 4. Inicializar DINO SDK com Configurações Databricks

Agora vamos inicializar o DINO SDK e testar suas funcionalidades.

In [ ]:
# Inicializar DINO SDK
print("🦕 Inicializando DINO SDK v1.2.0")
print("=" * 40)

# Importar DINO SDK
try:
    from src import (
        configure_dino_sdk,
        validate_dino_config,
        show_dino_config,
        create_unity_catalog_schema
    )
    print("✅ DINO SDK importado com sucesso!")
    
    # Verificar versão
    from src import __version__
    print(f"✅ Versão: {__version__}")
    
except Exception as e:
    print(f"❌ Erro ao importar DINO SDK: {e}")

# Configurar DINO SDK para Databricks
try:
    workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
    
    config_result = configure_dino_sdk(
        workspace_url=workspace_url,
        catalog_name="main",  # Usar seu catálogo aqui
        checkpoint_base_path="/tmp/checkpoints/dino_sdk",
        volume_base_path="/Volumes"
    )
    
    print("✅ DINO SDK configurado para Databricks!")
    print(f"   Workspace: {workspace_url}")
    
except Exception as e:
    print(f"❌ Erro na configuração do DINO SDK: {e}")

## 5. Validar Configuração Databricks

Vamos validar que a configuração está funcionando corretamente.

In [ ]:
# Validar configuração
print("🔍 Validando Configuração DINO SDK")
print("=" * 40)

# Mostrar configuração atual
try:
    show_dino_config()
except Exception as e:
    print(f"❌ Erro ao mostrar configuração: {e}")

print("\n" + "=" * 40)

# Validar configuração
try:
    validation_result = validate_dino_config()
    print("✅ Validação da configuração concluída!")
except Exception as e:
    print(f"❌ Erro na validação: {e}")

## 6. Testar Criação de Schema com DINO SDK

Agora vamos testar a funcionalidade principal: criação de schemas no Unity Catalog.

In [ ]:
# Testar criação de schema
print("📊 Testando Criação de Schema")
print("=" * 35)

# Definir parâmetros do teste
test_catalog = "main"  # Altere para seu catálogo
test_schema = "dino_test_schema"

print(f"🎯 Catálogo: {test_catalog}")
print(f"🎯 Schema: {test_schema}")

# Método 1: Usar função do DINO SDK
print("\n📋 Método 1: Usando função DINO SDK")
try:
    result = create_unity_catalog_schema(
        catalog_name=test_catalog,
        schema_name=test_schema
    )
    print(f"✅ Resultado: {result}")
except Exception as e:
    print(f"❌ Erro com função DINO SDK: {e}")

# Método 2: Usar código direto (fallback)
print("\n📋 Método 2: Usando código direto")
try:
    # Verificar se catálogo existe
    spark.sql(f"DESCRIBE CATALOG {test_catalog}").collect()
    print(f"✅ Catálogo '{test_catalog}' encontrado")
    
    # Obter external location
    external_location_result = spark.sql(f"DESCRIBE EXTERNAL LOCATION {test_catalog}").select("url").collect()
    if external_location_result:
        external_location = external_location_result[0].url
        schema_location = f"{external_location}/{test_catalog}/{test_schema}/"
        
        print(f"✅ External Location: {external_location}")
        print(f"✅ Schema Location: {schema_location}")
        
        # Criar schema
        spark.sql(f"""
            CREATE SCHEMA IF NOT EXISTS {test_catalog}.{test_schema}
            MANAGED LOCATION '{schema_location}'
        """)
        
        print(f"✅ Schema '{test_catalog}.{test_schema}' criado com sucesso!")
        
        # Verificar se foi criado
        schemas = spark.sql(f"SHOW SCHEMAS IN {test_catalog}").collect()
        schema_names = [row.schemaName for row in schemas]
        
        if test_schema in schema_names:
            print(f"✅ Schema verificado na lista de schemas")
        else:
            print(f"⚠️ Schema não encontrado na lista")
            
    else:
        print(f"❌ External location não encontrada para catálogo '{test_catalog}'")
        
except Exception as e:
    print(f"❌ Erro com código direto: {e}")

## 7. Testar Comando CLI do DINO SDK

Finalmente, vamos testar o comando CLI do DINO SDK.

In [ ]:
# Testar comando CLI
print("💻 Testando Comando CLI")
print("=" * 25)

# Testar comando dino-config setup
test_project = "teste_dino"
test_storage = "teststorage"
test_catalog_cli = "main"  # Altere para seu catálogo
test_schema_cli = "dino_cli_test"

print(f"🎯 Projeto: {test_project}")
print(f"🎯 Storage: {test_storage}")
print(f"🎯 Catálogo: {test_catalog_cli}")
print(f"🎯 Schema: {test_schema_cli}")

# Executar comando CLI
try:
    # Usar ! para executar comando shell
    command = f"dino-config setup --project-name {test_project} --storage-name {test_storage} --catalog-name {test_catalog_cli} --schema-name {test_schema_cli}"
    print(f"\n📋 Executando: {command}")
    
    result = dbutils.notebook.run("/path/to/notebook", timeout_seconds=60, arguments={
        "command": command
    })
    
    print(f"✅ Comando CLI executado")
    
except Exception as e:
    print(f"❌ Erro no comando CLI: {e}")
    print("💡 Tente executar manualmente:")
    print(f"   !{command}")

## 8. Resultado Final

Resumo dos testes e próximos passos.

In [ ]:
# Resultado final
print("🎉 DINO SDK v1.2.0 - Teste Concluído")
print("=" * 45)

# Verificar schemas criados
try:
    schemas = spark.sql(f"SHOW SCHEMAS IN {test_catalog}").collect()
    print(f"📊 Schemas no catálogo '{test_catalog}':")
    for schema in schemas:
        schema_name = schema.schemaName
        if 'dino' in schema_name.lower():
            print(f"   ✅ {schema_name} (criado pelo DINO SDK)")
        else:
            print(f"   📋 {schema_name}")
            
except Exception as e:
    print(f"❌ Erro ao listar schemas: {e}")

print("\n📝 Próximos Passos:")
print("1. ✅ DINO SDK v1.2.0 instalado e configurado")
print("2. ✅ Ambiente Databricks detectado corretamente")
print("3. ✅ Sessão Spark configurada")
print("4. ✅ Schemas criados no Unity Catalog")
print("5. 📋 Usar DINO SDK para ingestão de dados")
print("6. 📋 Configurar pipelines de dados")

print("\n💡 Comandos úteis:")
print("   dino-config show              # Mostrar configuração")
print("   dino-config validate          # Validar configuração")
print("   dino-ingest --help           # Ajuda para ingestão")

print("\n🦕 DINO SDK v1.2.0 está pronto para uso!")